# Assignment 1: Constraint Mapping Across Data Sources

This notebook maps power market constraints across three sources: PJMISO Market data, Dayzer, and Panorama. The objective is to identify which entries appear to represent the same underlying physical constraint, despite differences in naming conventions and formatting.

The final required output is a CSV with three columns:

- market_constraint
- dayzer_constraint
- pano_constraint

In [2]:
import pandas as pd
import numpy as np
import re
from rapidfuzz import fuzz, process
import matplotlib.pyplot as plt

print("Packages loaded successfully")

Packages loaded successfully


## 1. Load the Data

The assignment provides three constraint lists:

- PJMISO Market data
- Dayzer
- Panorama

Each source uses a different structure. Market and Panorama include separate facility and contingency information, while Dayzer mainly provides a single constraint name field.

In [3]:
market_path = "../data/Market PJMISO constraint list.csv"
dayzer_path = "../data/dayzer PJMISO constraint list.csv"
pano_path = "../data/Pano PJMISO constraint list.csv"

market_df = pd.read_csv(market_path)
dayzer_df = pd.read_csv(dayzer_path)
pano_df = pd.read_csv(pano_path)

print("Market shape:", market_df.shape)
print("Dayzer shape:", dayzer_df.shape)
print("Pano shape:", pano_df.shape)

Market shape: (5230, 6)
Dayzer shape: (13813, 2)
Pano shape: (21963, 5)


## 2. Preview the Data

Before building the matching logic, I previewed each dataset to understand the available columns and how each source describes constraints.

In [4]:
display(market_df.head())
display(dayzer_df.head())
display(pano_df.head())

,CONSTRAINT,CONTINGENCY,TOZONE,REPORTEDNAME,CONSTRAINTID,CONTINGENCYID
0,NOTTINGH 230 KV NOTTINGHM 2-3 SER DEV,L500.CONASTONE-PEACHBOTTOM.5012,NaN,NOTTINGHM 2-3 SER DEV A 230 KV,10002384305,10002493264
1,LENOX 115 KV LENOX-NMESHOPP NML 1090,L230.ETOWANDA-HILLSIDE.2002 [NYISO],PENELEC,LENOX-NMESHOPP NML 1090 B 115 KV,10000566138,10001865201
2,EASTON 69 KV EAS-EMU,ACTUAL,DPL,EASTON 69 KV EAS-EMU,10004072634,10000680485
3,SAYRECON230 KV SAY-SAY,ACTUAL,JCPL,SAYRECON230 KV SAY-SAY,10001822928,10000680485
4,MOUN UGI 230 KV MOUN UGI 2 XFORMER,230/66.MOUNTAIN.T1 (SCTNLZ),NaN,MOUN UGI230 KV 2,10000465752,10016012236


,CID,NAME
0,1,Eastern Interface
1,2,Central Interface
2,3,Western Interface
3,4,BCPEP Interface
4,6,Black Oak - Bedington Interface


,PID,Monitored Facility,Contingency Name,Earliest,Latest
0,1,CHIC_AVE 138KV - PRAXAIR3 138KV (CHIC_AVE 138 ...,L765.Dumont-WiltonCenter.11215,2017-06-29 00:00:00-04:00,2026-04-28 23:00:00-04:00
1,2,177 BURN 345KV - MUNSTER2 345KV (177 BURN 345 ...,L765.Dumont-WiltonCenter.11215,2016-11-23 03:00:00-05:00,2026-04-28 23:00:00-04:00
2,3,CONASTON 500KV - CONASTON 230KV (CONASTON 500 ...,L500.Brighton-Conastone.5011,2021-02-17 06:00:00-05:00,2026-04-28 23:00:00-04:00
3,4,SALTSPRG 138KV - MASURY 138KV (SALTSPRG 138 KV...,L345.Niles-Shenango,2025-11-03 09:40:00-05:00,2026-04-28 23:00:00-04:00
4,5,BURMA 115KV - PINEY 115KV (BURMA 115 KV BUR-PIN),L230.Glade-Warren.2088,2022-07-06 11:00:00-04:00,2026-04-28 23:00:00-04:00


## 3. Create Comparable Constraint Strings

To compare the three datasets, I created one readable constraint string for each source.

For Market data, I combined the `CONSTRAINT` and `CONTINGENCY` fields.

For Dayzer, I used the `NAME` field.

For Panorama, I combined the `Monitored Facility` and `Contingency Name` fields.

In [5]:
# Make copies so we do not accidentally change the original data
market = market_df.copy()
dayzer = dayzer_df.copy()
pano = pano_df.copy()

# Create one readable constraint string for each source
market["market_constraint"] = (
    market["CONSTRAINT"].fillna("").astype(str).str.strip()
    + " | "
    + market["CONTINGENCY"].fillna("").astype(str).str.strip()
)

dayzer["dayzer_constraint"] = dayzer["NAME"].fillna("").astype(str).str.strip()

pano["pano_constraint"] = (
    pano["Monitored Facility"].fillna("").astype(str).str.strip()
    + " | "
    + pano["Contingency Name"].fillna("").astype(str).str.strip()
)

display(market[["market_constraint"]].head())
display(dayzer[["dayzer_constraint"]].head())
display(pano[["pano_constraint"]].head())

,market_constraint
0,NOTTINGH 230 KV NOTTINGHM 2-3 SER DEV | L500.C...
1,LENOX 115 KV LENOX-NMESHOPP NML 1090 | L230.ET...
2,EASTON 69 KV EAS-EMU | ACTUAL
3,SAYRECON230 KV SAY-SAY | ACTUAL
4,MOUN UGI 230 KV MOUN UGI 2 XFORMER | 230/66.MO...


,dayzer_constraint
0,Eastern Interface
1,Central Interface
2,Western Interface
3,BCPEP Interface
4,Black Oak - Bedington Interface


,pano_constraint
0,CHIC_AVE 138KV - PRAXAIR3 138KV (CHIC_AVE 138 ...
1,177 BURN 345KV - MUNSTER2 345KV (177 BURN 345 ...
2,CONASTON 500KV - CONASTON 230KV (CONASTON 500 ...
3,SALTSPRG 138KV - MASURY 138KV (SALTSPRG 138 KV...
4,BURMA 115KV - PINEY 115KV (BURMA 115 KV BUR-PI...


In [6]:
print("Market rows:", len(market))
print("Dayzer rows:", len(dayzer))
print("Pano rows:", len(pano))

print("Unique Market constraints:", market["market_constraint"].nunique())
print("Unique Dayzer constraints:", dayzer["dayzer_constraint"].nunique())
print("Unique Pano constraints:", pano["pano_constraint"].nunique())

Market rows: 5230
Dayzer rows: 13813
Pano rows: 21963
Unique Market constraints: 5230
Unique Dayzer constraints: 13777
Unique Pano constraints: 21963


## 4. Normalize Text for Matching

The same physical constraint may be written differently across sources. To improve matching quality, I normalized the text by:

- converting everything to lowercase
- standardizing voltage formatting
- removing punctuation and extra separators
- replacing common abbreviations such as `xfmr` with `transformer`
- removing extra spaces

In [8]:
def normalize_constraint_text(text):
    """
    Normalize constraint text so different naming formats are easier to compare.
    This function:
    - lowercases text
    - removes extra punctuation
    - standardizes voltage formatting
    - removes extra spaces
    """
    if pd.isna(text):
        return ""
    
    text = str(text).lower()
    
    # Standardize common voltage formats
    text = re.sub(r'(\d+)\s*kv', r'\1kv', text)
    
    # Replace separators with spaces
    text = re.sub(r'[\|\-\_\/\(\)\[\]\.,;:]', ' ', text)
    
    # Remove extra non-alphanumeric characters
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    
    # Standardize common abbreviations/terms
    replacements = {
        "xfmr": "transformer",
        "xformer": "transformer",
        "transf": "transformer",
        "ckt": "circuit",
        "cir": "circuit",
        "ln": "line",
        "tx": "transformer",
    }
    
    for old, new in replacements.items():
        text = re.sub(rf'\b{old}\b', new, text)
    
    # Collapse repeated spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

## 5. Apply Normalization

After defining the normalization function, I applied it to each source so the matching algorithm compares cleaned versions of the constraint names instead of raw text.

In [9]:
market["market_clean"] = market["market_constraint"].apply(normalize_constraint_text)
dayzer["dayzer_clean"] = dayzer["dayzer_constraint"].apply(normalize_constraint_text)
pano["pano_clean"] = pano["pano_constraint"].apply(normalize_constraint_text)

display(market[["market_constraint", "market_clean"]].head())
display(dayzer[["dayzer_constraint", "dayzer_clean"]].head())
display(pano[["pano_constraint", "pano_clean"]].head())

,market_constraint,market_clean
0,NOTTINGH 230 KV NOTTINGHM 2-3 SER DEV | L500.C...,nottingh 230kv nottinghm 2 3 ser dev l500 cona...
1,LENOX 115 KV LENOX-NMESHOPP NML 1090 | L230.ET...,lenox 115kv lenox nmeshopp nml 1090 l230 etowa...
2,EASTON 69 KV EAS-EMU | ACTUAL,easton 69kv eas emu actual
3,SAYRECON230 KV SAY-SAY | ACTUAL,sayrecon230kv say say actual
4,MOUN UGI 230 KV MOUN UGI 2 XFORMER | 230/66.MO...,moun ugi 230kv moun ugi 2 transformer 230 66 m...


,dayzer_constraint,dayzer_clean
0,Eastern Interface,eastern interface
1,Central Interface,central interface
2,Western Interface,western interface
3,BCPEP Interface,bcpep interface
4,Black Oak - Bedington Interface,black oak bedington interface


,pano_constraint,pano_clean
0,CHIC_AVE 138KV - PRAXAIR3 138KV (CHIC_AVE 138 ...,chic ave 138kv praxair3 138kv chic ave 138kv c...
1,177 BURN 345KV - MUNSTER2 345KV (177 BURN 345 ...,177 burn 345kv munster2 345kv 177 burn 345kv b...
2,CONASTON 500KV - CONASTON 230KV (CONASTON 500 ...,conaston 500kv conaston 230kv conaston 500kv 5...
3,SALTSPRG 138KV - MASURY 138KV (SALTSPRG 138 KV...,saltsprg 138kv masury 138kv saltsprg 138kv sal...
4,BURMA 115KV - PINEY 115KV (BURMA 115 KV BUR-PI...,burma 115kv piney 115kv burma 115kv bur pin l2...


## 6. Review Cleaning Examples

I checked examples from each dataset to confirm that the normalization function was producing readable and comparable cleaned text.

In [10]:
print("Example cleaned Market constraint:")
print(market.loc[0, "market_constraint"])
print("→")
print(market.loc[0, "market_clean"])

print("\nExample cleaned Dayzer constraint:")
print(dayzer.loc[0, "dayzer_constraint"])
print("→")
print(dayzer.loc[0, "dayzer_clean"])

print("\nExample cleaned Pano constraint:")
print(pano.loc[0, "pano_constraint"])
print("→")
print(pano.loc[0, "pano_clean"])

Example cleaned Market constraint:
NOTTINGH 230 KV NOTTINGHM 2-3 SER DEV | L500.CONASTONE-PEACHBOTTOM.5012
→
nottingh 230kv nottinghm 2 3 ser dev l500 conastone peachbottom 5012

Example cleaned Dayzer constraint:
Eastern Interface
→
eastern interface

Example cleaned Pano constraint:
CHIC_AVE 138KV - PRAXAIR3 138KV (CHIC_AVE 138 KV CHI-PRA2) | L765.Dumont-WiltonCenter.11215
→
chic ave 138kv praxair3 138kv chic ave 138kv chi pra2 l765 dumont wiltoncenter 11215


## 7. Fuzzy Matching Function

I used fuzzy string matching to compare each cleaned Market constraint against the cleaned Dayzer and Panorama constraint lists.

The function returns the closest matching constraint and a similarity score. Higher scores indicate stronger text similarity.

In [11]:
def find_best_match(query, choices, scorer=fuzz.token_set_ratio):
    """
    Finds the best fuzzy match for one cleaned constraint string.
    """
    if not query:
        return None, 0
    
    result = process.extractOne(
        query,
        choices,
        scorer=scorer
    )
    
    if result is None:
        return None, 0
    
    best_match, score, index = result
    return best_match, score

print("Matching function ready")

Matching function ready


## 8. Build Lookup Lists

The fuzzy matching algorithm compares cleaned text, but the final output should show the original readable constraint names. These lookup dictionaries allow the notebook to match on cleaned text and then return the original text.

In [12]:
# These let us match using cleaned text, then return the original readable text
dayzer_lookup = dict(zip(dayzer["dayzer_clean"], dayzer["dayzer_constraint"]))
pano_lookup = dict(zip(pano["pano_clean"], pano["pano_constraint"]))

dayzer_choices = dayzer["dayzer_clean"].dropna().unique().tolist()
pano_choices = pano["pano_clean"].dropna().unique().tolist()

print("Dayzer choices:", len(dayzer_choices))
print("Pano choices:", len(pano_choices))

Dayzer choices: 13775
Pano choices: 21864


## 9. Test Matching on a Small Sample

Before running the matching process on the full Market dataset, I tested the logic on the first 10 Market constraints. This helped confirm that the matching approach was producing reasonable Dayzer and Panorama matches.

In [13]:
test_market = market.head(10).copy()

test_results = []

for _, row in test_market.iterrows():
    market_original = row["market_constraint"]
    market_clean = row["market_clean"]
    
    best_dayzer_clean, dayzer_score = find_best_match(market_clean, dayzer_choices)
    best_pano_clean, pano_score = find_best_match(market_clean, pano_choices)
    
    test_results.append({
        "market_constraint": market_original,
        "dayzer_constraint": dayzer_lookup.get(best_dayzer_clean, ""),
        "dayzer_score": dayzer_score,
        "pano_constraint": pano_lookup.get(best_pano_clean, ""),
        "pano_score": pano_score
    })

test_results_df = pd.DataFrame(test_results)

display(test_results_df)

,market_constraint,dayzer_constraint,dayzer_score,pano_constraint,pano_score
0,NOTTINGH 230 KV NOTTINGHM 2-3 SER DEV | L500.C...,NOTTINGH_230 KV_2-3:L500.Conastone-PeachBottom...,100.000000,NOTTINGHM 2-3 SER DEV A 230 KV | L500.C...,98.333333
1,LENOX 115 KV LENOX-NMESHOPP NML 1090 | L230.ET...,LENOX_115 KV_LEN-NME:L230.ETowanda-Hillside.20...,91.836735,NMESHOPP 115KV - LENOX 115KV (LENOX 115 KV LEN...,93.103448
2,EASTON 69 KV EAS-EMU | ACTUAL,EASTON_69 KV_EAS-EMU:ACTUAL,100.000000,EASTON 69KV - EMUNI 69KV (EASTON 69 KV EAS-EMU...,84.444444
3,SAYRECON230 KV SAY-SAY | ACTUAL,SAYRECON_230 KV_SAY-SAY:ACTUAL,77.551020,LINE 230 KV CED-ROSB | SAYREVIL230 KV E ...,70.270270
4,MOUN UGI 230 KV MOUN UGI 2 XFORMER | 230/66.MO...,MOUN UGI_230 KV_2:230/66.Mountain.T1 (Sctnlz),100.000000,MOUN UGI 69KV - MOUN UGI 230KV (MOUN UGI 230 K...,94.382022
5,GRACETON 230 KV GRACETON-SAFEHARB 2303 | L500....,GRACETON_230 KV_GRA-MANO:L500.Conastone-PeachB...,91.089109,GRACETON-SAFEHARB 2303 B 230 KV | (NUKE)...,100.000000
6,94 HAURD-11323 11323 B 138 KV | L345.NELSON-EL...,94 HAURD_138 KV_11323:L345.Nelson-Electric Jun...,92.592593,94 HAURD 138KV - 11323 138KV (94 HAURD 138 KV ...,100.000000
7,BERGEN 230 KV BER-HUD | ACTUAL,BERGEN_230 KV_BER-HUD:ACTUAL,100.000000,BERGEN 230KV - HCS 230KV (BERGEN 230 KV BER-HU...,85.106383
8,LINE 69 KV MONR AE-VINELAND 0711-1 | L230.CUMB...,MILL_69 KV_MIL-TUC:L230.Cumberland-Orchard.2314,83.544304,MONR AE 69KV - VINELAND 69KV (MONR AE 69 KV MO...,91.743119
9,GARDNERS 115 KV GARDNERS-TEXSEAST GAR-TEX | L1...,GARDNERS_115 KV_GAR-TEX:L115.MiddletownJct-Col...,100.000000,TEXSEAST 115KV - GARDNERS 115KV (GARDNERS 115 ...,100.000000


## 10. Review Small Sample Matches

I printed the first 10 matches in a readable format to manually inspect whether the selected Dayzer and Panorama matches made sense before applying the method to the full dataset.

In [14]:
for i, row in test_results_df.iterrows():
    print("=" * 100)
    print("MARKET:")
    print(row["market_constraint"])
    
    print("\nBEST DAYZER MATCH:")
    print(row["dayzer_constraint"])
    print("Dayzer score:", row["dayzer_score"])
    
    print("\nBEST PANO MATCH:")
    print(row["pano_constraint"])
    print("Pano score:", row["pano_score"])

MARKET:
NOTTINGH 230 KV NOTTINGHM 2-3 SER DEV | L500.CONASTONE-PEACHBOTTOM.5012

BEST DAYZER MATCH:
NOTTINGH_230 KV_2-3:L500.Conastone-PeachBottom.5012
Dayzer score: 100.0

BEST PANO MATCH:
NOTTINGHM 2-3 SER DEV       A  230 KV | L500.Conastone-PeachBottom.5012
Pano score: 98.33333333333333
MARKET:
LENOX 115 KV LENOX-NMESHOPP NML 1090 | L230.ETOWANDA-HILLSIDE.2002 [NYISO]

BEST DAYZER MATCH:
LENOX_115 KV_LEN-NME:L230.ETowanda-Hillside.2002 [NYISO]
Dayzer score: 91.83673469387755

BEST PANO MATCH:
NMESHOPP 115KV - LENOX 115KV (LENOX 115 KV LEN-NME) | L230.ETowanda-Hillside.2002 [NYISO]
Pano score: 93.10344827586206
MARKET:
EASTON  69 KV   EAS-EMU | ACTUAL

BEST DAYZER MATCH:
EASTON_69 KV_EAS-EMU:ACTUAL
Dayzer score: 100.0

BEST PANO MATCH:
EASTON 69KV - EMUNI 69KV (EASTON 69 KV EAS-EMU) | L69.Easton-EastonTap-Muni-Longwood-WyeMills.6707
Pano score: 84.44444444444444
MARKET:
SAYRECON230 KV SAY-SAY | ACTUAL

BEST DAYZER MATCH:
SAYRECON_230 KV_SAY-SAY:ACTUAL
Dayzer score: 77.55102040816327

## 11. Confidence Thresholds

To avoid forcing poor matches, I classified each match using score-based confidence levels:

- 90 to 100: high confidence
- 80 to 89: medium confidence
- 70 to 79: low confidence
- below 70: weak confidence

For the final output, I kept high and medium confidence matches and blanked out low or weak matches.

In [15]:
def confidence_label(score):
    """
    Categorize match quality based on fuzzy score.
    """
    if score >= 90:
        return "high"
    elif score >= 80:
        return "medium"
    elif score >= 70:
        return "low"
    else:
        return "weak"

test_results_df["dayzer_confidence"] = test_results_df["dayzer_score"].apply(confidence_label)
test_results_df["pano_confidence"] = test_results_df["pano_score"].apply(confidence_label)

display(test_results_df)

,market_constraint,dayzer_constraint,dayzer_score,pano_constraint,pano_score,dayzer_confidence,pano_confidence
0,NOTTINGH 230 KV NOTTINGHM 2-3 SER DEV | L500.C...,NOTTINGH_230 KV_2-3:L500.Conastone-PeachBottom...,100.000000,NOTTINGHM 2-3 SER DEV A 230 KV | L500.C...,98.333333,high,high
1,LENOX 115 KV LENOX-NMESHOPP NML 1090 | L230.ET...,LENOX_115 KV_LEN-NME:L230.ETowanda-Hillside.20...,91.836735,NMESHOPP 115KV - LENOX 115KV (LENOX 115 KV LEN...,93.103448,high,high
2,EASTON 69 KV EAS-EMU | ACTUAL,EASTON_69 KV_EAS-EMU:ACTUAL,100.000000,EASTON 69KV - EMUNI 69KV (EASTON 69 KV EAS-EMU...,84.444444,high,medium
3,SAYRECON230 KV SAY-SAY | ACTUAL,SAYRECON_230 KV_SAY-SAY:ACTUAL,77.551020,LINE 230 KV CED-ROSB | SAYREVIL230 KV E ...,70.270270,low,low
4,MOUN UGI 230 KV MOUN UGI 2 XFORMER | 230/66.MO...,MOUN UGI_230 KV_2:230/66.Mountain.T1 (Sctnlz),100.000000,MOUN UGI 69KV - MOUN UGI 230KV (MOUN UGI 230 K...,94.382022,high,high
5,GRACETON 230 KV GRACETON-SAFEHARB 2303 | L500....,GRACETON_230 KV_GRA-MANO:L500.Conastone-PeachB...,91.089109,GRACETON-SAFEHARB 2303 B 230 KV | (NUKE)...,100.000000,high,high
6,94 HAURD-11323 11323 B 138 KV | L345.NELSON-EL...,94 HAURD_138 KV_11323:L345.Nelson-Electric Jun...,92.592593,94 HAURD 138KV - 11323 138KV (94 HAURD 138 KV ...,100.000000,high,high
7,BERGEN 230 KV BER-HUD | ACTUAL,BERGEN_230 KV_BER-HUD:ACTUAL,100.000000,BERGEN 230KV - HCS 230KV (BERGEN 230 KV BER-HU...,85.106383,high,medium
8,LINE 69 KV MONR AE-VINELAND 0711-1 | L230.CUMB...,MILL_69 KV_MIL-TUC:L230.Cumberland-Orchard.2314,83.544304,MONR AE 69KV - VINELAND 69KV (MONR AE 69 KV MO...,91.743119,medium,high
9,GARDNERS 115 KV GARDNERS-TEXSEAST GAR-TEX | L1...,GARDNERS_115 KV_GAR-TEX:L115.MiddletownJct-Col...,100.000000,TEXSEAST 115KV - GARDNERS 115KV (GARDNERS 115 ...,100.000000,high,high


## 12. Review Questionable Sample Matches

After adding confidence labels to the small sample, I reviewed any matches that were not classified as high confidence. This helped confirm that lower-scoring matches should be handled carefully in the final output.

In [16]:
questionable_test = test_results_df[
    (test_results_df["dayzer_confidence"] != "high") |
    (test_results_df["pano_confidence"] != "high")
]

display(questionable_test)

,market_constraint,dayzer_constraint,dayzer_score,pano_constraint,pano_score,dayzer_confidence,pano_confidence
2,EASTON 69 KV EAS-EMU | ACTUAL,EASTON_69 KV_EAS-EMU:ACTUAL,100.000000,EASTON 69KV - EMUNI 69KV (EASTON 69 KV EAS-EMU...,84.444444,high,medium
3,SAYRECON230 KV SAY-SAY | ACTUAL,SAYRECON_230 KV_SAY-SAY:ACTUAL,77.551020,LINE 230 KV CED-ROSB | SAYREVIL230 KV E ...,70.270270,low,low
7,BERGEN 230 KV BER-HUD | ACTUAL,BERGEN_230 KV_BER-HUD:ACTUAL,100.000000,BERGEN 230KV - HCS 230KV (BERGEN 230 KV BER-HU...,85.106383,high,medium
8,LINE 69 KV MONR AE-VINELAND 0711-1 | L230.CUMB...,MILL_69 KV_MIL-TUC:L230.Cumberland-Orchard.2314,83.544304,MONR AE 69KV - VINELAND 69KV (MONR AE 69 KV MO...,91.743119,medium,high


## 13. Confidence Thresholds

To avoid forcing poor matches, I classified each match using score-based confidence levels:

- 90 to 100: high confidence
- 80 to 89: medium confidence
- 70 to 79: low confidence
- below 70: weak confidence

For the final output, I kept high and medium confidence matches and blanked out low or weak matches.

In [17]:
def confidence_label(score):
    """
    Converts a match score into an easy confidence label.
    """
    if score >= 90:
        return "high"
    elif score >= 80:
        return "medium"
    elif score >= 70:
        return "low"
    else:
        return "weak"

print("Confidence label function ready")

Confidence label function ready


## 14. Run Full Matching

After validating the method on a small sample, I applied the matching process to all Market constraints. For each Market constraint, the code selected the closest Dayzer match and the closest Panorama match, then assigned match scores and confidence labels.

In [18]:
full_results = []

for i, row in market.iterrows():
    market_original = row["market_constraint"]
    market_clean = row["market_clean"]
    
    best_dayzer_clean, dayzer_score = find_best_match(market_clean, dayzer_choices)
    best_pano_clean, pano_score = find_best_match(market_clean, pano_choices)
    
    full_results.append({
        "market_constraint": market_original,
        "dayzer_constraint": dayzer_lookup.get(best_dayzer_clean, ""),
        "dayzer_score": dayzer_score,
        "dayzer_confidence": confidence_label(dayzer_score),
        "pano_constraint": pano_lookup.get(best_pano_clean, ""),
        "pano_score": pano_score,
        "pano_confidence": confidence_label(pano_score)
    })
    
    # This prints progress every 500 rows so you know it is working
    if (i + 1) % 500 == 0:
        print(f"Processed {i + 1} of {len(market)} rows")

full_results_df = pd.DataFrame(full_results)

print("Full matching complete")
print("Total matched rows:", len(full_results_df))

display(full_results_df.head())

Processed 500 of 5230 rows
Processed 1000 of 5230 rows
Processed 1500 of 5230 rows
Processed 2000 of 5230 rows
Processed 2500 of 5230 rows
Processed 3000 of 5230 rows
Processed 3500 of 5230 rows
Processed 4000 of 5230 rows
Processed 4500 of 5230 rows
Processed 5000 of 5230 rows
Full matching complete
Total matched rows: 5230


,market_constraint,dayzer_constraint,dayzer_score,dayzer_confidence,pano_constraint,pano_score,pano_confidence
0,NOTTINGH 230 KV NOTTINGHM 2-3 SER DEV | L500.C...,NOTTINGH_230 KV_2-3:L500.Conastone-PeachBottom...,100.000000,high,NOTTINGHM 2-3 SER DEV A 230 KV | L500.C...,98.333333,high
1,LENOX 115 KV LENOX-NMESHOPP NML 1090 | L230.ET...,LENOX_115 KV_LEN-NME:L230.ETowanda-Hillside.20...,91.836735,high,NMESHOPP 115KV - LENOX 115KV (LENOX 115 KV LEN...,93.103448,high
2,EASTON 69 KV EAS-EMU | ACTUAL,EASTON_69 KV_EAS-EMU:ACTUAL,100.000000,high,EASTON 69KV - EMUNI 69KV (EASTON 69 KV EAS-EMU...,84.444444,medium
3,SAYRECON230 KV SAY-SAY | ACTUAL,SAYRECON_230 KV_SAY-SAY:ACTUAL,77.551020,low,LINE 230 KV CED-ROSB | SAYREVIL230 KV E ...,70.270270,low
4,MOUN UGI 230 KV MOUN UGI 2 XFORMER | 230/66.MO...,MOUN UGI_230 KV_2:230/66.Mountain.T1 (Sctnlz),100.000000,high,MOUN UGI 69KV - MOUN UGI 230KV (MOUN UGI 230 K...,94.382022,high


## 15. Review Match Quality

After running the full matching process, I summarized the confidence levels for both Dayzer and Panorama matches. This helped evaluate how many matches were strong, moderate, or questionable.

In [19]:
print("Dayzer confidence counts:")
display(full_results_df["dayzer_confidence"].value_counts())

print("Pano confidence counts:")
display(full_results_df["pano_confidence"].value_counts())

Dayzer confidence counts:


dayzer_confidence
high      2885
medium    1982
low        286
weak        77
Name: count, dtype: int64

Pano confidence counts:


pano_confidence
high      2708
medium    1770
low        403
weak       349
Name: count, dtype: int64

## 16. Review Questionable Full Matches

I separated matches where either the Dayzer or Panorama result was not high confidence. This review table helps identify constraints that may require manual checking or should not be forced into the final output.

In [20]:
questionable_matches = full_results_df[
    (full_results_df["dayzer_confidence"] != "high") |
    (full_results_df["pano_confidence"] != "high")
].copy()

print("Number of questionable matches:", len(questionable_matches))

display(questionable_matches.head(25))

Number of questionable matches: 3141


,market_constraint,dayzer_constraint,dayzer_score,dayzer_confidence,pano_constraint,pano_score,pano_confidence
2,EASTON 69 KV EAS-EMU | ACTUAL,EASTON_69 KV_EAS-EMU:ACTUAL,100.000000,high,EASTON 69KV - EMUNI 69KV (EASTON 69 KV EAS-EMU...,84.444444,medium
3,SAYRECON230 KV SAY-SAY | ACTUAL,SAYRECON_230 KV_SAY-SAY:ACTUAL,77.551020,low,LINE 230 KV CED-ROSB | SAYREVIL230 KV E ...,70.270270,low
7,BERGEN 230 KV BER-HUD | ACTUAL,BERGEN_230 KV_BER-HUD:ACTUAL,100.000000,high,BERGEN 230KV - HCS 230KV (BERGEN 230 KV BER-HU...,85.106383,medium
8,LINE 69 KV MONR AE-VINELAND 0711-1 | L230.CUMB...,MILL_69 KV_MIL-TUC:L230.Cumberland-Orchard.2314,83.544304,medium,MONR AE 69KV - VINELAND 69KV (MONR AE 69 KV MO...,91.743119,high
13,BERWICK 69 KV BERWICK-KOONSVIL | ACTUAL,KOONSVIL_69 KV_XFMR1:ACTUAL,86.956522,medium,BERWICK-KOONSVIL A 69 KV | BASE,85.714286,medium
14,PREST TIBB 138 KV L/O PRAIRIE STATE MT VERNON ...,PRES_138 KV_PRE-TIBB:Prairie State-W Mt. Verno...,89.887640,medium,PRES 138KV - TIBB 138KV (PRES 138 KV PRE-TIBB)...,89.887640,medium
17,DOEX530 345 KV 203T | ACTUAL,DOEX530_345 KV_203T:ACTUAL,100.000000,high,DOEX530 13.8KV - DOEX530 345KV (DOEX530 345 KV...,83.720930,medium
18,DOEX530 345 KV 208T | ACTUAL,DOEX530_345 KV_208T:ACTUAL,100.000000,high,DOEX530 345 KV 208T | BASE,87.804878,medium
19,BOONETWN230 KV BOO-SRE | 500/230.LAUSCHTOWN.T3,BOONETWN_230 KV_BOO-SRE:500/230.Lauschtown.T3,87.356322,medium,BOONETWN 230KV - SREADING 230KV (BOONETWN 230 ...,80.555556,medium
22,94 HAURD138 KV 96101 | ACTUAL,94 HAURD_138 KV_96101:ACTUAL,79.245283,low,94 HAURD138 KV 18623 | BASE,70.270270,low


## 17. Create Final Output

For the final required CSV, I kept high and medium confidence matches and blanked out low or weak matches. This avoids forcing uncertain matches into the final answer.

The final output includes only the three required columns:

- market_constraint
- dayzer_constraint
- pano_constraint

In [21]:
final_results = full_results_df.copy()

# Blank out low-quality Dayzer matches
final_results.loc[final_results["dayzer_score"] < 80, "dayzer_constraint"] = ""

# Blank out low-quality Pano matches
final_results.loc[final_results["pano_score"] < 80, "pano_constraint"] = ""

# Keep only the required columns for the final CSV
final_output = final_results[[
    "market_constraint",
    "dayzer_constraint",
    "pano_constraint"
]].copy()

display(final_output.head(20))

,market_constraint,dayzer_constraint,pano_constraint
0,NOTTINGH 230 KV NOTTINGHM 2-3 SER DEV | L500.C...,NOTTINGH_230 KV_2-3:L500.Conastone-PeachBottom...,NOTTINGHM 2-3 SER DEV A 230 KV | L500.C...
1,LENOX 115 KV LENOX-NMESHOPP NML 1090 | L230.ET...,LENOX_115 KV_LEN-NME:L230.ETowanda-Hillside.20...,NMESHOPP 115KV - LENOX 115KV (LENOX 115 KV LEN...
2,EASTON 69 KV EAS-EMU | ACTUAL,EASTON_69 KV_EAS-EMU:ACTUAL,EASTON 69KV - EMUNI 69KV (EASTON 69 KV EAS-EMU...
3,SAYRECON230 KV SAY-SAY | ACTUAL,,
4,MOUN UGI 230 KV MOUN UGI 2 XFORMER | 230/66.MO...,MOUN UGI_230 KV_2:230/66.Mountain.T1 (Sctnlz),MOUN UGI 69KV - MOUN UGI 230KV (MOUN UGI 230 K...
5,GRACETON 230 KV GRACETON-SAFEHARB 2303 | L500....,GRACETON_230 KV_GRA-MANO:L500.Conastone-PeachB...,GRACETON-SAFEHARB 2303 B 230 KV | (NUKE)...
6,94 HAURD-11323 11323 B 138 KV | L345.NELSON-EL...,94 HAURD_138 KV_11323:L345.Nelson-Electric Jun...,94 HAURD 138KV - 11323 138KV (94 HAURD 138 KV ...
7,BERGEN 230 KV BER-HUD | ACTUAL,BERGEN_230 KV_BER-HUD:ACTUAL,BERGEN 230KV - HCS 230KV (BERGEN 230 KV BER-HU...
8,LINE 69 KV MONR AE-VINELAND 0711-1 | L230.CUMB...,MILL_69 KV_MIL-TUC:L230.Cumberland-Orchard.2314,MONR AE 69KV - VINELAND 69KV (MONR AE 69 KV MO...
9,GARDNERS 115 KV GARDNERS-TEXSEAST GAR-TEX | L1...,GARDNERS_115 KV_GAR-TEX:L115.MiddletownJct-Col...,TEXSEAST 115KV - GARDNERS 115KV (GARDNERS 115 ...


## 18. Save Final CSV

I saved the required assignment output as `matched_constraints.csv` in the output folder. This file contains the required columns only:

- market_constraint
- dayzer_constraint
- pano_constraint

In [22]:
final_output.to_csv("../output/matched_constraints.csv", index=False)

print("Saved final CSV to: ../output/matched_constraints.csv")

Saved final CSV to: ../output/matched_constraints.csv


## 19. Save Scored Review File

I also saved a scored review file that includes match scores and confidence labels. This file is useful for transparency, quality control, and explaining the matching method in the summary report.

In [23]:
full_results_df.to_csv("../output/matched_constraints_with_scores.csv", index=False)

print("Saved review CSV to: ../output/matched_constraints_with_scores.csv")

Saved review CSV to: ../output/matched_constraints_with_scores.csv


## 20. Save Questionable Matches File

I saved a separate file containing questionable matches for review. These are cases where either the Dayzer or Panorama match was not high confidence, so they may require manual inspection.

In [24]:
questionable_matches.to_csv("../output/questionable_matches_for_review.csv", index=False)

print("Saved questionable matches to: ../output/questionable_matches_for_review.csv")

Saved questionable matches to: ../output/questionable_matches_for_review.csv


## 21. Summary Statistics

I created a summary table to show how many Dayzer and Panorama matches fell into each confidence category. This also shows how many matches were kept in the final output and how many were blanked out due to lower confidence.

In [25]:
# Summary statistics for the matching results

total_rows = len(full_results_df)

dayzer_kept = (full_results_df["dayzer_score"] >= 80).sum()
pano_kept = (full_results_df["pano_score"] >= 80).sum()

dayzer_blank = total_rows - dayzer_kept
pano_blank = total_rows - pano_kept

summary_stats = pd.DataFrame({
    "Source": ["Dayzer", "Panorama"],
    "High Confidence": [
        (full_results_df["dayzer_confidence"] == "high").sum(),
        (full_results_df["pano_confidence"] == "high").sum()
    ],
    "Medium Confidence": [
        (full_results_df["dayzer_confidence"] == "medium").sum(),
        (full_results_df["pano_confidence"] == "medium").sum()
    ],
    "Low Confidence": [
        (full_results_df["dayzer_confidence"] == "low").sum(),
        (full_results_df["pano_confidence"] == "low").sum()
    ],
    "Weak Confidence": [
        (full_results_df["dayzer_confidence"] == "weak").sum(),
        (full_results_df["pano_confidence"] == "weak").sum()
    ],
    "Kept in Final Output": [dayzer_kept, pano_kept],
    "Blanked in Final Output": [dayzer_blank, pano_blank]
})

display(summary_stats)

,Source,High Confidence,Medium Confidence,Low Confidence,Weak Confidence,Kept in Final Output,Blanked in Final Output
0,Dayzer,2885,1982,286,77,4867,363
1,Panorama,2708,1770,403,349,4478,752


## 22. Summary Percentages

I converted the summary counts into percentages to make the match quality easier to interpret. This helps show the share of constraints that had strong, moderate, or questionable matches.

In [26]:
summary_percent = summary_stats.copy()

for col in ["High Confidence", "Medium Confidence", "Low Confidence", "Weak Confidence", "Kept in Final Output", "Blanked in Final Output"]:
    summary_percent[col] = (summary_percent[col] / total_rows * 100).round(2).astype(str) + "%"

display(summary_percent)

,Source,High Confidence,Medium Confidence,Low Confidence,Weak Confidence,Kept in Final Output,Blanked in Final Output
0,Dayzer,55.16%,37.9%,5.47%,1.47%,93.06%,6.94%
1,Panorama,51.78%,33.84%,7.71%,6.67%,85.62%,14.38%


## 23. Save Summary Statistics

I saved the summary statistics table as a CSV file so the match quality results can be referenced in the report and included in the final project output.

In [27]:
summary_stats.to_csv("../output/match_summary_statistics.csv", index=False)

print("Saved summary statistics to: ../output/match_summary_statistics.csv")

Saved summary statistics to: ../output/match_summary_statistics.csv
